In [ ]:
# Name: Ansik Singh Tomar
# Roll No: 2401037

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, trim
import numpy as np
import pandas as pd

# Create Spark session
spark = SparkSession.builder \
    .appName("Assignment9") \
    .master("local[*]") \
    .getOrCreate()

In [ ]:

# Define dataset columns
columns = [
    "age", "workclass", "fnlwgt", "education",
    "education_num", "marital_status", "occupation",
    "relationship", "race", "sex", "capital_gain",
    "capital_loss", "hours_per_week", "native_country",
    "income"
]

# Read dataset
df = spark.read.csv(
    "adult.data",
    header=False,
    inferSchema=True,
    ignoreLeadingWhiteSpace=True
)

df = df.toDF(*columns)

# Remove extra spaces
for name, dtype in df.dtypes:
    if dtype == "string":
        df = df.withColumn(name, trim(col(name)))

print("Dataset:")
df.show(5)
print("Total Records:", df.count())

In [ ]:
# QUESTION 1: Manual K-Means Clustering

# Select the required attributes and remove missing values
cluster_df = df.filter(
    (col("workclass") != "?") &
    (col("education") != "?")
)

data = pd.DataFrame([row.asDict() for row in cluster_df.collect()])

# Convert categorical attributes into numerical values
workclass_values = sorted(data["workclass"].unique())
education_values = sorted(data["education"].unique())

workclass_map = {value: index for index, value in enumerate(workclass_values)}
education_map = {value: index for index, value in enumerate(education_values)}

workclass_encoded = np.eye(len(workclass_values))[
    data["workclass"].map(workclass_map).values
]

education_encoded = np.eye(len(education_values))[
    data["education"].map(education_map).values
]

# Normalize age
age = data["age"].values.astype(float)
age = (age - age.mean()) / age.std()

# Combine all three attributes
X = np.column_stack((age, workclass_encoded, education_encoded))

# Initialize K-Means
k = 3
max_iterations = 100

np.random.seed(42)
centroid_indices = np.random.choice(len(X), k, replace=False)
centroids = X[centroid_indices].copy()

previous_labels = np.full(len(X), -1)

# K-Means algorithm
for iteration in range(max_iterations):

    # Calculate distance from each point to each centroid
    distances = np.sqrt(
        ((X[:, np.newaxis, :] - centroids) ** 2).sum(axis=2)
    )

    # Assign points to nearest centroid
    labels = np.argmin(distances, axis=1)

    # Update centroids
    new_centroids = centroids.copy()

    for i in range(k):
        cluster_points = X[labels == i]

        if len(cluster_points) > 0:
            new_centroids[i] = cluster_points.mean(axis=0)

    # Check convergence
    if np.array_equal(labels, previous_labels):
        centroids = new_centroids
        break

    previous_labels = labels.copy()
    centroids = new_centroids

# Store cluster labels
distances = np.sqrt(
    ((X[:, np.newaxis, :] - centroids) ** 2).sum(axis=2)
)

data["cluster"] = np.argmin(distances, axis=1)

# Display results
print("QUESTION 1: K-MEANS CLUSTERING RESULTS")
print(data[["age", "workclass", "education", "cluster"]].head(10))

print("\nCLUSTER COUNTS:")
print(data["cluster"].value_counts().sort_index())

print("\nTotal clustered records:", len(data))
print("Iterations performed:", iteration + 1)

# Save clustered data
data.to_csv("clustered_adult.csv", index=False)
print("\nOutput saved as clustered_adult.csv")

In [ ]:
# Convert DataFrame into RDD
rdd = df.rdd

# QUESTION 2 (i): Country with the highest number of adults, excluding USA

country_count = rdd.filter(
    lambda row: row.native_country not in ["United-States", "?"]
).map(
    lambda row: (row.native_country, 1)
).reduceByKey(
    lambda a, b: a + b
).sortBy(
    lambda x: x[1], ascending=False
)

print("QUESTION 2 (i):")
print("Country with highest number of adults:", country_count.take(1))


# QUESTION 2 (ii): Number of people with at least Master's education
# working in Tech-support

masters_count = rdd.filter(
    lambda row:
    row.education_num >= 14 and
    row.occupation == "Tech-support"
).count()

print("\nQUESTION 2 (ii):")
print("Number of people:", masters_count)


# QUESTION 2 (iii): Number of unmarried males working in Local-gov

unmarried_count = rdd.filter(
    lambda row:
    row.sex == "Male" and
    row.marital_status == "Never-married" and
    row.workclass == "Local-gov"
).count()

print("\nQUESTION 2 (iii):")
print("Number of unmarried males:", unmarried_count)

# Stop Spark only after all questions are completed
spark.stop()